# Chapter 1 Lab: The Imperative of Performance in Modern AI

**Dr. Kaoutar El Maghraoui**

Chapter 1 sets out why performance is a design requirement, defines the four factors of AI system performance (latency, throughput, cost and reliability) and introduces the RAG case study. This notebook measures those ideas on the hardware you have.

By the end you will have:

1. An auto-detected **hardware tier**, one of the four capability classes of Table 1.2 (consumer CPU, consumer GPU 16 GB, datacenter GPU 40 GB, datacenter GPU 80 GB+).
2. A measured **latency-throughput curve** for a small matmul workload on your GPU, the trade-off that Figure 1.4 shows for the case-study generator.
3. A working **naïve RAG baseline** with the architecture of Figure 1.10 (*System architecture overview*): an embedding model, a FAISS index and a small instruct LM.
4. An end-to-end **latency breakdown** for a single query, with the stages of Table 1.3 and Figure 1.11 (query encoding, retrieval, context assembly, prefill, decode).
5. A **cost-per-query** projection: the same model and answers, measured as a serial stream and projected to large query volumes.
6. Answers to the **Knowledge check** at the end of the chapter (also in `knowledge_check_solutions.md`).

### Before you start

Set **Runtime → Change runtime type → Hardware accelerator: T4 GPU.** The notebook runs on CPU too, but generation is slower: the single-query cell took 12.5 s on a 2-vCPU cloud CPU against 2.6 s on a T4, and on CPU the waterfall is dominated by CPU-bound matmul instead of the bottleneck the chapter is about.

Total runtime, measured on Modal in November 2026 with the tested stack: 73 s on a T4 and 252 s on a 2-vCPU cloud CPU, plus the model downloads on a fresh machine. Timings on your machine will differ, most of all on CPU, where the host processor sets the speed.

## 0  Setup

Install the stack. The `-q` flag keeps the output tidy. About 90 seconds on a cold Colab runtime.

In [ ]:
# Tested stack for the book (Chapters 1-5 and 7): Python 3.11, PyTorch 2.14.1 (CUDA 13.0),
# transformers 5.20.2, sentence-transformers 6.2.0. Colab ships its own PyTorch build with CUDA,
# so torch is not reinstalled here (installing an older wheel would downgrade the runtime).
# The library versions match the Chapter 2 lab, so nothing has to be reinstalled later.
!pip install -q transformers==5.20.2 sentence-transformers==6.2.0 accelerate faiss-cpu datasets bitsandbytes

## 1  Hardware tier auto-detection (Table 1.2)

Table 1.2 organizes the case study by capability class, because capability classes change less often than product names. The pipeline is the same in every class; only the model sizes change. The cell below picks the tier for the machine you are running on, and the rest of the notebook uses that tier's models.

In [ ]:
import torch, subprocess, sys, platform

def detect_tier():
    """Return one of: cpu, gpu_16gb, gpu_40gb, gpu_80gb (Table 1.2 capability classes)."""
    if not torch.cuda.is_available():
        return 'cpu'
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram_gb < 24:    return 'gpu_16gb'   # T4 (15 GB), L4 (24 GB rounded down), RTX 4060 Ti 16 GB
    if vram_gb < 60:    return 'gpu_40gb'   # A100 40 GB, L40 (48 GB)
    return 'gpu_80gb'                      # A100 80 GB, H100 80 GB, H200 141 GB

TIER = detect_tier()

# Table 1.2 model recipe per tier. Model names are 2026 exemplars;
# the structural property (same pipeline, scaled model size) is what
# matters and persists as exemplars rotate.
RECIPE = {
    'cpu':      {'embed': 'sentence-transformers/all-MiniLM-L6-v2', 'embed_dim': 384,  'lm': 'Qwen/Qwen2.5-0.5B-Instruct',     'lm_4bit': False, 'expected_ttft_s': '10-30'},
    'gpu_16gb': {'embed': 'sentence-transformers/all-MiniLM-L6-v2', 'embed_dim': 384,  'lm': 'Qwen/Qwen2.5-1.5B-Instruct',     'lm_4bit': True,  'expected_ttft_s': '3-8'},
    'gpu_40gb': {'embed': 'BAAI/bge-base-en-v1.5',                  'embed_dim': 768,  'lm': 'Qwen/Qwen2.5-3B-Instruct',       'lm_4bit': False, 'expected_ttft_s': '1-3'},
    'gpu_80gb': {'embed': 'BAAI/bge-large-en-v1.5',                 'embed_dim': 1024, 'lm': 'Qwen/Qwen2.5-7B-Instruct',       'lm_4bit': False, 'expected_ttft_s': '0.5-1'},
}[TIER]

print(f'Detected tier: {TIER}')
print(f'Python:        {sys.version.split()[0]}')
print(f'PyTorch:       {torch.__version__}')
print(f'Platform:      {platform.platform()}')
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f'GPU:           {torch.cuda.get_device_name(0)}  ({p.total_memory/1e9:.1f} GB HBM, {p.multi_processor_count} SMs, sm_{p.major}{p.minor})')
else:
    print('GPU:           none. Falling back to CPU recipe')
print()
print(f'Embedding model:    {RECIPE["embed"]}  ({RECIPE["embed_dim"]}-d)')
print(f'Generation model:   {RECIPE["lm"]}{"  (4-bit)" if RECIPE["lm_4bit"] else ""}')
print(f'Expected naïve TTFT for tier {TIER}: ~{RECIPE["expected_ttft_s"]} s')

If you got `gpu_16gb` on Colab, you are on the smallest GPU tier that runs the case-study labs of Chapters 1 to 5 (the row marked ♦ in Table 1.2). This notebook uses the tier's suggested models (all-MiniLM-L6-v2 and Qwen2.5-1.5B-Instruct in 4-bit on that tier), which are smaller than the configuration measured in the chapter (BGE-M3 and Gemma-2B-IT in FP16). Its milliseconds are therefore not comparable with Table 1.3; compare the order of the stages and their shares.

## 2  The latency-throughput trade-off (*Throughput*, Figure 1.4)

Figure 1.4 plots decode step time against throughput for Gemma-2B-IT in FP16 at batch sizes 1, 4, 16 and 64 on three GPUs. On the T4, going from batch size 1 to 64 raised decode throughput from 38 to 1,050 tokens/s and each decode step from 26 to 61 ms; on the A100 and H100 the step time stayed almost constant up to batch size 64.

The cells below measure the same kind of trade-off on your GPU with a small MLP (768 → 3,072 → 768) that stands in for one transformer block, swept from batch size 1 to 64: wall-clock latency per request and aggregate throughput. The shape of the curve is what to look at; the absolute numbers depend on your tier.

In [ ]:
import time, torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# A request = one forward pass through a fake 768→3072→768 MLP.
# Roughly the per-token cost of a single transformer block at small scale.
torch.manual_seed(0)
W1 = torch.randn(768, 3072, device=device, dtype=torch.float16 if device=='cuda' else torch.float32)
W2 = torch.randn(3072, 768, device=device, dtype=torch.float16 if device=='cuda' else torch.float32)

def step(batch_size):
    x = torch.randn(batch_size, 768, device=device, dtype=W1.dtype)
    return torch.relu(x @ W1) @ W2

def benchmark(batch_size, n_iter=20, warmup=3):
    for _ in range(warmup):
        y = step(batch_size)
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        y = step(batch_size)
    if device == 'cuda': torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    batch_latency_ms = (elapsed / n_iter) * 1000
    per_request_latency_ms = batch_latency_ms                    # in a batched system every request waits for the full batch
    throughput_rps = batch_size / (batch_latency_ms / 1000)
    return per_request_latency_ms, throughput_rps

batch_sizes = [1, 2, 4, 8, 16, 32, 64]
results = [(bs, *benchmark(bs)) for bs in batch_sizes]

print(f'{"BS":>4}  {"per-req latency (ms)":>22}  {"throughput (req/s)":>20}')
for bs, lat, tput in results:
    print(f'{bs:>4}  {lat:>22.2f}  {tput:>20.1f}')

# --- self-check: throughput rises with batch size and timings are sane ---
_tputs = [tp for _, _, tp in results]
_lats  = [la for _, la, _ in results]
assert all(v > 0 for v in _tputs + _lats), 'benchmark produced a non-positive timing'
assert _tputs[-1] > _tputs[0], f'throughput should rise with batch size, got {_tputs}'
print(f'self-check passed: throughput {_tputs[0]:.0f} -> {_tputs[-1]:.0f} req/s across batch sizes')


In [ ]:
# Plot the curve. Three illustrative operating zones are shaded; the chapter's Figure 1.4 has no zones.
import matplotlib.pyplot as plt

lat = [r[1] for r in results]
tput = [r[2] for r in results]

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tput, lat, 'o-', linewidth=2, color='#B85C2C', markersize=8, zorder=5)
for (bs, lat_v, tput_v) in results:
    ax.annotate(f'BS={bs}', (tput_v, lat_v), textcoords='offset points', xytext=(8, 8), fontsize=9)

# Illustrative operating zones (latency bands in ms), a reading aid for this notebook.
ymax = max(max(lat) * 1.15, 1.0)
# Zone boundaries clamped to the visible range so a fast GPU
# with sub-millisecond latencies does not push labels off the axis.
zones = [
    (0,   75,   'green', 0.10, 'Real-time (fraud, search)',    '#1f6b1f'),
    (75,  150,  'gold',  0.10, 'Interactive (chatbots)',       '#7a5a00'),
    (150, ymax, 'red',   0.08, 'Batch (indexing, offline)',    '#7a1f1f'),
]
for lo, hi, color, alpha, label, tcolor in zones:
    if lo >= ymax:
        continue
    hi = min(hi, ymax)
    ax.axhspan(lo, hi, alpha=alpha, color=color, zorder=0)
    ax.text(max(tput)*0.98, (lo + hi) / 2, label, ha='right', va='center', fontsize=9, color=tcolor)

ax.set_xlabel('Throughput (requests / second)')
ax.set_ylabel('Per-request latency (ms)')
ax.set_title('Latency-throughput trade-off on your hardware  (reproduces Figure 1.4)')
ax.set_ylim(0, ymax)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Read the curve. Throughput rises with batch size because one weight read serves every request in the batch. Per-request latency stays almost flat while the batch is smaller than what the GPU processes in the same time, and rises after that point. Compare how much throughput and latency grew between batch size 1 and 64 on your GPU.

The shaded zones are a reading aid for this notebook; the chapter's Figure 1.4 has none. The operating points form the throughput-latency **Pareto frontier** of the *Throughput* section, and Chapter 8 covers how to choose a point on it.

## 3  Build the naïve RAG baseline (*System architecture overview*)

Now the case-study system itself. **This is the unoptimized baseline** that motivates every chapter that follows. Three pieces, matching Figure 1.10:

1. **Indexing (offline):** chunk a small corpus, embed each chunk, store in FAISS.
2. **Retrieval (online):** encode the query with the same embedding model, ANN-search the top-k.
3. **Generation (online):** prompt the LM with the retrieved chunks, generate an answer.

The corpus is the book's lab corpus: 2,000 WikiText-2 passages, the same passages the Chapter 2 lab uses. all-MiniLM-L6-v2 reads at most 256 tokens of each passage. The timing cell after the index build measures how long indexing takes on your hardware.

In [ ]:
from sentence_transformers import SentenceTransformer
from datasets import load_dataset
import faiss, numpy as np

# Lab corpus: 2,000 WikiText-2 passages of 200 to 800 characters, the same passages as the Chapter 2 lab.
wiki = load_dataset('Salesforce/wikitext', 'wikitext-2-raw-v1', split='train')
corpus = [t.strip() for t in wiki['text'] if 200 < len(t) < 800][:2000]
print(f'Corpus loaded: {len(corpus)} passages, mean length {int(np.mean([len(p.split()) for p in corpus]))} tokens')

embedder = SentenceTransformer(RECIPE['embed'], device=device)
embeddings = embedder.encode(corpus, batch_size=32, show_progress_bar=True, convert_to_numpy=True, normalize_embeddings=True)

index = faiss.IndexFlatIP(RECIPE['embed_dim'])   # exact inner-product search over all vectors
index.add(embeddings.astype(np.float32))
print(f'FAISS index: {index.ntotal} vectors, dim={RECIPE["embed_dim"]}')

### Indexing time for the book's corpus

The chapter's lab corpus is 2,000 WikiText-2 passages. The cell below times all-MiniLM-L6-v2 on the corpus you just indexed: embedding at batch sizes 32 and 128, then the FAISS index build. Run it once on a CPU runtime and once on a T4 to get the speedup the chapter describes. The model loads before the timer starts, so the first download is not counted.

In [ ]:
import time, statistics

passages = corpus                                # the 2,000 WikiText-2 passages indexed above

minilm = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device=device)
minilm.max_seq_length = 256                      # the model's input limit; longer passages are truncated
n_tok = [len(x) for x in minilm.tokenizer(passages, truncation=True, max_length=256)['input_ids']]
print(f'{len(passages)} passages, mean {statistics.mean(n_tok):.0f} tokens, '
      f'{sum(t >= 256 for t in n_tok)} truncated at 256')
minilm.encode(passages[:8], batch_size=8)        # warm-up

index_times = {}
for bs in (32, 128):
    if device == 'cuda': torch.cuda.synchronize()
    t0 = time.perf_counter()
    emb = minilm.encode(passages, batch_size=bs, convert_to_numpy=True, normalize_embeddings=True)
    if device == 'cuda': torch.cuda.synchronize()
    t_embed = time.perf_counter() - t0
    t0 = time.perf_counter()
    idx = faiss.IndexFlatIP(emb.shape[1]); idx.add(emb.astype(np.float32))
    t_index = time.perf_counter() - t0
    index_times[bs] = t_embed + t_index
    print(f'batch {bs:>3}: embed {t_embed:6.2f} s ({len(passages)/t_embed:7.1f} passages/s), '
          f'FAISS add {t_index*1000:.1f} ms, total {t_embed + t_index:6.2f} s on {device}')

# --- self-check: every passage is in the index ---
assert idx.ntotal == len(passages), f'expected {len(passages)} vectors, got {idx.ntotal}'
print(f'self-check passed: {idx.ntotal} vectors of dim {emb.shape[1]} indexed')

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

lm_kwargs = {'torch_dtype': torch.float16, 'device_map': 'auto'} if device == 'cuda' else {'torch_dtype': torch.float32}
if RECIPE['lm_4bit'] and device == 'cuda':
    from transformers import BitsAndBytesConfig
    lm_kwargs['quantization_config'] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

tok = AutoTokenizer.from_pretrained(RECIPE['lm'])
lm  = AutoModelForCausalLM.from_pretrained(RECIPE['lm'], **lm_kwargs)
if device == 'cpu': lm = lm.to('cpu')
lm.eval()
print(f'Loaded {RECIPE["lm"]}')

In [ ]:
@torch.inference_mode()
def naive_rag(query, top_k=3, max_new_tokens=64):
    """Naïve baseline. Every stage is unoptimized on purpose. That is the point of Chapter 1."""
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    _, idx = index.search(q_emb, top_k)
    retrieved = [corpus[i] for i in idx[0]]
    context = '\n\n'.join(f'[{i+1}] {p}' for i, p in enumerate(retrieved))
    prompt = f'Use the context to answer.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:'
    inputs = tok(prompt, return_tensors='pt').to(lm.device)
    out = lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    answer = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return answer

answer = naive_rag('Who wrote The Importance of Being Earnest?')
print('Q: Who wrote The Importance of Being Earnest?')
print(f'A: {answer.strip()}')

## 4  The latency breakdown (Table 1.3, Figure 1.11)

Time each stage of the pipeline separately. Table 1.3 and Figure 1.11 give this breakdown for the chapter's configuration (BGE-M3 and Gemma-2B-IT in FP16, 512-token prompt, 20-token answer): 379 ms at the median on the A100, 189 ms on the H100 and 673 ms on the T4, with decode taking 89%, 88% and 76% of the sum of the stage medians. This notebook runs your tier's smaller models, so its milliseconds differ; compare the order of the stages and the decode share.

In [ ]:
import time

@torch.inference_mode()
def naive_rag_timed(query, top_k=3, max_new_tokens=20):
    timings = {}
    if device == 'cuda': torch.cuda.synchronize()

    # Stage 1: query encoding
    t = time.perf_counter()
    q_emb = embedder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    if device == 'cuda': torch.cuda.synchronize()
    timings['query_encode_ms'] = (time.perf_counter() - t) * 1000

    # Stage 2: ANN retrieval
    t = time.perf_counter()
    _, idx = index.search(q_emb, top_k)
    timings['retrieval_ms'] = (time.perf_counter() - t) * 1000

    # Stage 3: context assembly
    t = time.perf_counter()
    retrieved = [corpus[i] for i in idx[0]]
    context = '\n\n'.join(f'[{i+1}] {p}' for i, p in enumerate(retrieved))
    prompt = f'Use the context to answer.\n\nContext:\n{context}\n\nQuestion: {query}\nAnswer:'
    timings['context_assembly_ms'] = (time.perf_counter() - t) * 1000

    # Stage 4: LLM prefill (just the prompt forward pass)
    inputs = tok(prompt, return_tensors='pt').to(lm.device)
    if device == 'cuda': torch.cuda.synchronize()
    t = time.perf_counter()
    out_prefill = lm(**inputs)
    if device == 'cuda': torch.cuda.synchronize()
    timings['llm_prefill_ms'] = (time.perf_counter() - t) * 1000

    # Stage 5: LLM decode (generate N new tokens)
    t = time.perf_counter()
    out = lm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tok.eos_token_id)
    if device == 'cuda': torch.cuda.synchronize()
    timings['llm_decode_ms'] = (time.perf_counter() - t) * 1000

    timings['total_ms'] = sum(v for k, v in timings.items() if k.endswith('_ms'))
    timings['answer'] = tok.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    return timings

# Warmup. First call always carries CUDA-init / autotune costs that don't represent steady state.
_ = naive_rag_timed('What is the capital of Ireland?')

# Real measurement
t = naive_rag_timed('What is the capital of Ireland?')
print(f'Q: What is the capital of Ireland?')
print(f'A: {t["answer"]}')
print()
print(f'{"Stage":<24}{"Time (ms)":>12}    Bottleneck class       Primary technique (chapter)')
print('-' * 100)
rows = [
    ('Query encoding',       t['query_encode_ms'],     'Compute-bound',           'torch.compile, quantization (Ch. 3-4, 9)'),
    ('ANN retrieval',        t['retrieval_ms'],        'Memory-bound',            'Index optimization, GPU FAISS (Ch. 6)'),
    ('Context assembly',     t['context_assembly_ms'], 'I/O-bound',               'Caching, async prefetch (Ch. 8)'),
    ('LLM prefill',          t['llm_prefill_ms'],      'Compute-bound',           'FlashAttention, torch.compile (Ch. 4-5, 8)'),
    ('LLM decode (20 toks)', t['llm_decode_ms'],       'Memory-bandwidth bound',  'Speculative decoding, KV quant. (Ch. 8-9)'),
]
for stage, ms, btype, tech in rows:
    print(f'{stage:<24}{ms:>12.1f}    {btype:<22} {tech}')
print('-' * 100)
print(f'{"Total":<24}{t["total_ms"]:>12.1f}    LLM decode dominates'.ljust(80))
print(f'\nDecode share of total: {t["llm_decode_ms"] / t["total_ms"] * 100:.1f}%')

# --- self-check: decode dominates the naive pipeline ---
_stage_ms = {k: v for k, v in t.items() if k.endswith('_ms') and k != 'total_ms'}
assert t['total_ms'] > 0, 'total latency should be positive'
assert max(_stage_ms, key=_stage_ms.get) == 'llm_decode_ms', f'expected decode to dominate, got {_stage_ms}'
assert t['llm_decode_ms'] / t['total_ms'] > 0.3, f"decode share unexpectedly low: {t['llm_decode_ms']/t['total_ms']*100:.0f}%"
print(f"self-check passed: decode is the largest stage ({t['llm_decode_ms']/t['total_ms']*100:.0f}% of total)")


In [ ]:
# Visualize as a stacked bar. The chapter calls this a 'latency waterfall'.
import matplotlib.pyplot as plt

stages = ['Query\nencoding', 'ANN\nretrieval', 'Context\nassembly', 'LLM\nprefill', 'LLM\ndecode']
values = [t['query_encode_ms'], t['retrieval_ms'], t['context_assembly_ms'], t['llm_prefill_ms'], t['llm_decode_ms']]
colors = ['#9CC9E0', '#7FB3D5', '#F5B041', '#E67E22', '#B85C2C']

fig, ax = plt.subplots(figsize=(10, 2.5))
left = 0
for s, v, c in zip(stages, values, colors):
    ax.barh([0], [v], left=left, color=c, edgecolor='white', linewidth=1.5)
    if v > t['total_ms'] * 0.04:
        ax.text(left + v/2, 0, f'{s}\n{v:.0f} ms', ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    left += v
ax.set_xlim(0, t['total_ms'] * 1.02)
ax.set_xlabel('Time (ms)')
ax.set_yticks([])
ax.set_title(f'Naïve RAG latency waterfall, total {t["total_ms"]:.0f} ms on tier `{TIER}`  (reproduces structure of Table 1.2)')
plt.tight_layout()
plt.show()

Two things to notice.

**LLM decode is the largest stage.** In Table 1.3 decode takes the largest share of the latency on all three GPUs measured; check the decode share printed above for your tier. Vector retrieval takes under 1 ms in the chapter's configuration, so optimizing it first would leave the total almost unchanged (Principle #1, *Measure before you optimize*).

**Memory bandwidth sets the ceiling of each decode step.** *The GenAI inference problem* section explains why: each generated token reads all the model weights from GPU memory and performs about one floating-point operation per byte read. On the A100 the measured step was several times longer than that bandwidth floor, and Chapter 2 shows that per-step host overhead takes the rest. Compilation (Chapter 4), batching and speculative decoding (Chapter 8) and quantization (Chapter 9) target this stage.

## 5  Cost per query (*The accuracy–performance gap*, *Cost*)

The Booking.com and Microsoft Bing/BERT case studies in the chapter describe models that met their accuracy targets and were held back by latency or serving cost. The cell below runs the RAG system as a serial stream of queries, measures the cost per query at an hourly GPU price, and projects it to large daily query volumes. The model and the answers stay the same; only the volume changes.

In [ ]:
import time

queries = [
    'What is the chemical symbol for antimony?',
    'Which band recorded the album South of Heaven?',
    'Who won the Boat Race in 1900?',
    'Who created the Flower Fairies illustrations?',
    'Who wrote and drew The Crab with the Golden Claws?',
    'In which country is the Krak des Chevaliers?',
    'Which team did Michael Jordan win six NBA championships with?',
    'Who discovered the neutrino with Clyde Cowan?',
]

# Single-request mode (interactive)
if device == 'cuda': torch.cuda.synchronize()
t0 = time.perf_counter()
for q in queries:
    _ = naive_rag(q, max_new_tokens=20)
if device == 'cuda': torch.cuda.synchronize()
t_serial = time.perf_counter() - t0

# Naïvely 'batched'. We just measure throughput; real batching belongs to Ch. 8.
# This is a thought experiment about cost economics, not a prescription.
per_query_serial_s = t_serial / len(queries)
throughput_serial  = len(queries) / t_serial

# Use a hypothetical T4-on-Colab cost as a stand-in for the cloud GPU rate.
# The 'cost per useful output' calculation from Chapter 1's *Cost* section.
GPU_PRICE_PER_HOUR = 0.40           # example hourly price; set it to the price you pay
cost_per_query_usd = (per_query_serial_s / 3600) * GPU_PRICE_PER_HOUR
print(f'Serial run of {len(queries)} queries:')
print(f'  total wall-clock:       {t_serial:.2f} s')
print(f'  per-query latency:      {per_query_serial_s*1000:.0f} ms')
print(f'  throughput:             {throughput_serial:.2f} queries/s')
print(f'  cost per query:         ${cost_per_query_usd:.5f}  (at ${GPU_PRICE_PER_HOUR}/hr)')
print()
print(f'At 10 M queries/day (Bing-scale):  ${cost_per_query_usd * 10_000_000:,.0f}/day')
print(f'At 100 M queries/day (ChatGPT-scale): ${cost_per_query_usd * 100_000_000:,.0f}/day')

The dollar figures at the bottom scale linearly with the cost per query you measured and the hourly price you set. The chapter reports inference shares of 60 to 90% of ML compute in the cited reports (*Cost*, Figure 1.7), and the share grows for as long as a model stays in production, so a gain in serving efficiency recurs every day the model serves. None of the book's optimizations is applied yet; the later chapters measure what each one changes.

## 6  Knowledge check

The six questions of the chapter's Knowledge check. Try answering them in your own words before you open the suggested answers. The same answers are in `knowledge_check_solutions.md`.

---

**Q1.** A fraud detection model achieves 98.2% accuracy but requires 500 ms per inference, and the SLA requires sub-50 ms response time. (a) How would you frame this as a performance engineering problem? (b) What are the first two diagnostic questions you would ask before attempting any optimization, and why?

<details><summary>Suggested answer</summary>

**Answer.**(a) The model meets the accuracy requirement and misses the latency requirement by a factor of ten, so
the task is to reach the 50 ms target at the same accuracy. (b) First, where do the 500 ms go, stage by stage?
Second, what is the full specification: the percentile and load at which 50 ms must hold, and the throughput,
cost and reliability targets?

**Explanation.** The chapter calls this distance between what a model achieves in evaluation and what it can
deliver under production constraints the accuracy–performance gap, and Table 1.1 lists real-time fraud detection
with a critical latency priority (*Translating lab accuracy to production value*, *The accuracy–performance gap*).
The two questions are Principle #1, measure before you optimize, and Principle #2, define targets first
(*Business constraints and real-world trade-offs*): without the stage breakdown the work may go to a stage that is
not the bottleneck, and without the specification there is no stopping point.

</details>

---

**Q2.** Explain the difference between latency and throughput in an LLM serving system. Describe a scenario where optimizing for throughput increases latency, and the mechanism.

<details><summary>Suggested answer</summary>

**Answer.**Latency is the time of one request, from arrival to the last byte of the response, and is described by
its distribution (p50, p95, p99). Throughput is the work the whole system completes per unit time, in requests or
tokens per second. Batching decode steps raises throughput and, past a GPU-dependent batch size, lengthens every
request's decode steps.

**Explanation.** On the T4, going from batch size 1 to 64 raised Gemma-2B-IT decode throughput from 38 to 1,050
tokens/s while each decode step grew from 26 to 61 ms, so every request in the batch waited longer (*Throughput*,
Figure 1.4). With static batching a request also waits until its batch is formed, and each step slows once the
batch exceeds what the GPU processes at a constant step time (*The interdependence of the four factors*).

</details>

---

**Q3.** Rank the four performance factors for a production chatbot serving 10,000 concurrent users, then for a nightly batch pipeline that summarizes 1 million documents. How does the ranking change, and why?

<details><summary>Suggested answer</summary>

**Answer.**Chatbot: latency and reliability first, then throughput and cost. Nightly batch: throughput and cost
first, then reliability, with latency last.

**Explanation.** Table 1.1 gives the interactive chatbot a high latency priority (under 2 s TTFT), high reliability
and moderate throughput and cost, because users see each request; it gives batch document work critical throughput
and cost priorities and a low latency priority, because only the completion of the whole job matters
(*The interdependence of the four factors*). The ranking follows from who waits for the result: a user waiting on
one request, or a schedule waiting on the whole batch.

</details>

---

**Q4.** The LLM decode stage dominates naive end-to-end latency (Table 1.3). Explain why memory bandwidth sets the lower bound on decode step time, why the measured A100 step is several times slower than that bound, and what this implies about the bottleneck.

<details><summary>Suggested answer</summary>

**Answer.**Each decode step reads all model weights from GPU memory to produce one token, so the step cannot be
faster than the weight bytes divided by the memory bandwidth: 5.0 GB of FP16 weights at the A100's peak bandwidth
take about 2.5 ms. The measured step takes about 17.7 ms in the lab's decode loop (16.5 ms with `generate()`),
because per-step host overhead adds most of the time. Decode is the stage to work on first, and on the A100 the
first target is that overhead.

**Explanation.** Decode at batch 1 performs about one floating-point operation per byte of weights read and ran at
about 15% of its bandwidth ceiling on the A100 (*The GenAI inference problem: Stateful and memory-hungry*,
Figure 1.12). Decode takes 89% of the sum of stage medians on the A100 and 76% to 88% on the H100 and T4, so the
generation stage comes before the retrieval engine and the embedding model (*Latency budget of the naive
pipeline*, Table 1.3); Chapter 2 attributes the rest of each step to host overhead.

</details>

---

**Q5.** Training GPT-4 is reported to have cost more than $100 million; take $100 million. At 10 million queries per day and $0.01 per query, how long before inference costs exceed training costs? What does this imply about engineering investment?

<details><summary>Suggested answer</summary>

**Answer.**Inference costs 10,000,000 × $0.01 = $100,000 per day, so it passes $100 million after 1,000 days,
about 2.7 years. Engineering effort on inference efficiency returns more than the same effort on training
efficiency for a model that stays in production.

**Explanation.** The chapter reports inference shares of 60% to 90% of ML compute in the cited reports and states
that a 2× improvement in inference efficiency is worth more than a 2× improvement in training efficiency, because
the inference saving recurs every day the model is in production and the training saving occurs once (*Cost*,
Figure 1.7). This is why most of the book's performance work targets serving (*The accuracy–performance gap*).

</details>

---

**Q6.** Identify two differences in compute characteristics that make GPU kernel optimization for ML workloads (for example, attention) different from kernel optimization for traditional HPC workloads (for example, finite-element simulation). Consider memory access patterns, data types and parallelism structure.

<details><summary>Suggested answer</summary>

**Answer.**Any two of the following. Data types: HPC codes have traditionally used FP64 for solver stability,
while ML training tolerates BF16 and accelerators deliver an order of magnitude more throughput in it (about 15× on
the H100, dense BF16 against FP64 Tensor Core peaks). Memory access: HPC kernels work on structured, regular grids
with predictable access, while attention is irregular and data-dependent and decode updates a growing KV cache at
every step. Parallelism: HPML adds tensor parallelism, which splits weight matrices, and pipeline parallelism,
which splits layers, to data parallelism.

**Explanation.** The irregularity makes classical HPC optimizations such as stencil computation and cache-oblivious
layouts only partly applicable, and the lower precision moves the limit of many inference kernels to memory
bandwidth, which grew about 21× from the V100 to Rubin against about 280× for peak Tensor Core throughput
(*HPML versus traditional HPC*, Figure 1.9).

</details>

## What you have just done

You measured the latency-throughput trade-off on your GPU, built the naïve RAG baseline, found the stage that dominates its latency and projected its cost per query.

Chapter 2 turns this stage timing into a methodology: the PyTorch profiler, Nsight Systems, the Roofline model, regression tests and a written performance specification. Chapter 3 measures the costs that eager PyTorch adds, Chapter 4 compiles the decode step, and Chapters 8 and 9 cover serving and quantization.

Save your end-to-end latency from Section 4. Chapter 2's regression test compares against it.

**Next:** open `chapter02_lab.ipynb`.